In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, explained_variance_score
from sklearn.model_selection import GridSearchCV
from matplotlib import pyplot as plt

In [3]:
rounding = 3

In [4]:
def print_feature_importances(cols, importances):
    idx = np.argsort(importances)[::-1]
    print(list(zip(np.array(cols)[idx], np.array(importances)[idx])))

def feature_importance_dict(cols, importances):
    dict = {}
    for i in range(len(cols)):
        col = cols[i]
        dict[col] = np.round(importances[i], rounding)
    return dict

In [11]:
df = pd.read_csv('../processed_data/processed_data.csv')
basin_dict = {"AL": 0.0, "CP": 1.0, "EP": 2.0}
for i in range(len(df)):
    df.at[i, "Basin"] = basin_dict[df.iloc[i]["Basin"]]
df["Basin"]

0       0.0
1       0.0
2       0.0
3       0.0
4       0.0
       ... 
1121    2.0
1122    2.0
1123    2.0
1124    2.0
1125    2.0
Name: Basin, Length: 1126, dtype: object

In [17]:
cols = ["logVMAX12", "MSLP12", "POT12", "VGRAD0-6", "VGRAD6-12", "ONI", "AL", "CP", "EP"]
X, y = df[cols].to_numpy(), df["logVMAX36"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=4)

rf = RandomForestRegressor()
# Grid search
params = {"n_estimators": [20, 50, 100],
          "criterion": ["squared_error", "friedman_mse", "absolute_error", "poisson"],
          "max_depth": [None, 3, 5, 6],
          "max_samples": [0.2, 0.5, 0.7],
          "min_samples_split": [2, 3, 5, 8],
          "max_features": [None, 2, 3, 5]}
rf_gs = GridSearchCV(rf, params, scoring="explained_variance", n_jobs=-1)

rf_gs.fit(X_train, y_train)

print("Best params:", rf_gs.best_params_)
print("Best score:", rf_gs.best_score_)

Best params: {'criterion': 'poisson', 'max_depth': 6, 'max_features': 5, 'max_samples': 0.7, 'min_samples_split': 2, 'n_estimators': 20}
Best score: 0.41496940195845333


In [23]:
rf = RandomForestRegressor(**{'criterion': 'poisson', 'max_depth': 6, 'max_features': 5, 'max_samples': 0.7, 'min_samples_split': 2, 'n_estimators': 20})
fit_rf = rf.fit(X_train, y_train)
rf_importances_df = pd.DataFrame({"Importance": feature_importance_dict(cols, fit_rf.feature_importances_)}).T
print(rf_importances_df.to_latex(caption="Feature Importances of sklearn Random Forest Regressor with Basin Indicators", label="tab:rf_importances"))
rf_importances_df

\begin{table}
\caption{Feature Importances of sklearn Random Forest Regressor with Basin Indicators}
\label{tab:rf_importances}
\begin{tabular}{lrrrrrrrrr}
\toprule
 & logVMAX12 & MSLP12 & POT12 & VGRAD0-6 & VGRAD6-12 & ONI & AL & CP & EP \\
\midrule
Importance & 0.423000 & 0.085000 & 0.129000 & 0.048000 & 0.190000 & 0.074000 & 0.008000 & 0.012000 & 0.031000 \\
\bottomrule
\end{tabular}
\end{table}



,logVMAX12,MSLP12,POT12,VGRAD0-6,VGRAD6-12,ONI,AL,CP,EP
Importance,0.423,0.085,0.129,0.048,0.19,0.074,0.008,0.012,0.031


In [24]:
y_pred = fit_rf.predict(X_test)
y_train_pred = fit_rf.predict(X_train)
out_of_sample = {"MSE": mean_squared_error(y_test, y_pred), "Explained Variance": explained_variance_score(y_test, y_pred)}
in_sample = {"MSE": mean_squared_error(y_train, y_train_pred), "Explained Variance": explained_variance_score(y_train, y_train_pred)}
rf_results = {"In Sample": in_sample, "Out of Sample": out_of_sample}
rf_results_df = pd.DataFrame(rf_results).T
print(rf_results_df.to_latex(caption="Scoring for sklearn Random Forest Regressor with Basin Indicators", label="tab:rf_scoring"))
rf_results_df

\begin{table}
\caption{Scoring for sklearn Random Forest Regressor with Basin Indicators}
\label{tab:rf_scoring}
\begin{tabular}{lrr}
\toprule
 & MSE & Explained Variance \\
\midrule
In Sample & 0.035999 & 0.562953 \\
Out of Sample & 0.041517 & 0.349450 \\
\bottomrule
\end{tabular}
\end{table}



,MSE,Explained Variance
In Sample,0.035999,0.562953
Out of Sample,0.041517,0.349450


In [32]:
cols = ["logVMAX12", "MSLP12", "POT12", "VGRAD0-6", "VGRAD6-12", "ONI", "AL", "CP", "EP"]
X, y = df[cols].to_numpy(), df["logVMAX36"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=4)

rf = RandomForestRegressor()
# Grid search
params = {"n_estimators": [50, 100, 120],
          "criterion": ["squared_error", "friedman_mse", "absolute_error", "poisson"],
          "max_depth": [3, 4, 5],
          "max_samples": [0.7],
          "min_samples_split": [2, 3],
          "max_features": [None, 2, 3, 6]}
rf_gs = GridSearchCV(rf, params, scoring="neg_mean_squared_error", n_jobs=-1)

rf_gs.fit(X_train, y_train)

print("Best params:", rf_gs.best_params_)
print("Best score:", rf_gs.best_score_)

Best params: {'criterion': 'squared_error', 'max_depth': 5, 'max_features': 3, 'max_samples': 0.7, 'min_samples_split': 3, 'n_estimators': 50}
Best score: -0.04918903759961992


In [33]:
rf = RandomForestRegressor(**rf_gs.best_params_)
fit_rf = rf.fit(X_train, y_train)
rf_importances_df = pd.DataFrame({"Importance": feature_importance_dict(cols, fit_rf.feature_importances_)}).T
print(rf_importances_df.to_latex(caption="Feature Importances of sklearn Random Forest Regressor with Basin Indicators", label="tab:rf_importances"))
rf_importances_df

\begin{table}
\caption{Feature Importances of sklearn Random Forest Regressor with Basin Indicators}
\label{tab:rf_importances}
\begin{tabular}{lrrrrrrrrr}
\toprule
 & logVMAX12 & MSLP12 & POT12 & VGRAD0-6 & VGRAD6-12 & ONI & AL & CP & EP \\
\midrule
Importance & 0.345000 & 0.165000 & 0.079000 & 0.063000 & 0.242000 & 0.057000 & 0.011000 & 0.011000 & 0.026000 \\
\bottomrule
\end{tabular}
\end{table}



,logVMAX12,MSLP12,POT12,VGRAD0-6,VGRAD6-12,ONI,AL,CP,EP
Importance,0.345,0.165,0.079,0.063,0.242,0.057,0.011,0.011,0.026


In [34]:
y_pred = fit_rf.predict(X_test)
y_train_pred = fit_rf.predict(X_train)
out_of_sample = {"MSE": mean_squared_error(y_test, y_pred), "Explained Variance": explained_variance_score(y_test, y_pred)}
in_sample = {"MSE": mean_squared_error(y_train, y_train_pred), "Explained Variance": explained_variance_score(y_train, y_train_pred)}
rf_results = {"In Sample": in_sample, "Out of Sample": out_of_sample}
rf_results_df = pd.DataFrame(rf_results).T
print(rf_results_df.to_latex(caption="Scoring for sklearn Random Forest Regressor with Basin Indicators", label="tab:rf_scoring"))
rf_results_df

\begin{table}
\caption{Scoring for sklearn Random Forest Regressor with Basin Indicators}
\label{tab:rf_scoring}
\begin{tabular}{lrr}
\toprule
 & MSE & Explained Variance \\
\midrule
In Sample & 0.041705 & 0.493686 \\
Out of Sample & 0.041050 & 0.358405 \\
\bottomrule
\end{tabular}
\end{table}



,MSE,Explained Variance
In Sample,0.041705,0.493686
Out of Sample,0.041050,0.358405


In [46]:
cols = ["logVMAX12", "MSLP12", "POT12", "VGRAD0-6", "VGRAD6-12", "ONI", "LAT", "LON"]
X, y = df[cols].to_numpy(), df["logVMAX36"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=int(np.floor(100*np.random.random())))

rf = RandomForestRegressor()
# Grid search
params = {"n_estimators": [20, 50, 100],
          "criterion": ["squared_error", "friedman_mse", "absolute_error", "poisson"],
          "max_depth": [None, 3, 5],
          "min_samples_split": [2, 3],
          "max_samples": [0.5, 0.7],
          "max_features": [None, 2, 3, 5]}
rf_gs = GridSearchCV(rf, params, scoring="neg_mean_squared_error", n_jobs=-1)

rf_gs.fit(X_train, y_train)

print("Best params:", rf_gs.best_params_)
print("Best score:", rf_gs.best_score_)

rf = RandomForestRegressor(**rf_gs.best_params_)
fit_rf = rf.fit(X_train, y_train)
latlon_rf_importances_df = pd.DataFrame({"Importance": feature_importance_dict(cols, fit_rf.feature_importances_)}).T
print(latlon_rf_importances_df.to_latex(caption="Feature Importances of sklearn Random Forest Regressor with Latitude and Longitude", label="tab:latlon_rf_importances"))

y_pred = fit_rf.predict(X_test)
y_train_pred = fit_rf.predict(X_train)
out_of_sample = {"MSE": mean_squared_error(y_test, y_pred), "Explained Variance": explained_variance_score(y_test, y_pred)}
in_sample = {"MSE": mean_squared_error(y_train, y_train_pred), "Explained Variance": explained_variance_score(y_train, y_train_pred)}
rf_results = {"In Sample": in_sample, "Out of Sample": out_of_sample}
latlon_rf_results_df = pd.DataFrame(rf_results).T
print(latlon_rf_results_df.to_latex(caption="Scoring for sklearn Random Forest Regressor with Latitude and Longitude", label="tab:latlon_rf_scoring"))

Best params: {'criterion': 'poisson', 'max_depth': 5, 'max_features': None, 'max_samples': 0.5, 'min_samples_split': 2, 'n_estimators': 50}
Best score: -0.04705564460954579
\begin{table}
\caption{Feature Importances of sklearn Random Forest Regressor with Latitude and Longitude}
\label{tab:latlon_rf_importances}
\begin{tabular}{lrrrrrrrr}
\toprule
 & logVMAX12 & MSLP12 & POT12 & VGRAD0-6 & VGRAD6-12 & ONI & LAT & LON \\
\midrule
Importance & 0.414000 & 0.045000 & 0.071000 & 0.010000 & 0.227000 & 0.047000 & 0.077000 & 0.108000 \\
\bottomrule
\end{tabular}
\end{table}

\begin{table}
\caption{Scoring for sklearn Random Forest Regressor with Latitude and Longitude}
\label{tab:latlon_rf_scoring}
\begin{tabular}{lrr}
\toprule
 & MSE & Explained Variance \\
\midrule
In Sample & 0.037331 & 0.514185 \\
Out of Sample & 0.045996 & 0.466459 \\
\bottomrule
\end{tabular}
\end{table}



In [47]:
latlon_rf_importances_df

,logVMAX12,MSLP12,POT12,VGRAD0-6,VGRAD6-12,ONI,LAT,LON
Importance,0.414,0.045,0.071,0.01,0.227,0.047,0.077,0.108


In [48]:
latlon_rf_results_df

,MSE,Explained Variance
In Sample,0.037331,0.514185
Out of Sample,0.045996,0.466459
